# 🧠 PINKCC : Entraînement ResNet18 (Méthode B + Advanced Augmentation)

Ce notebook teste la robustesse du modèle en combinant la correction du déséquilibre des classes (**Méthode B**) avec notre nouveau pipeline d'augmentation de données avancé (flou, contraste, rotation).

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from pathlib import Path
import matplotlib.pyplot as plt
import sys

# 1. Configuration du chemin pour importer src/
project_root = Path.cwd().parent
if str(project_root / "src") not in sys.path:
    sys.path.append(str(project_root / "src"))

# 2. Imports des modules PINKCC
from pinkcc_ct_seg.data.dataset import BrainMRIDataset
from pinkcc_ct_seg.data.split import get_train_val_splits
from pinkcc_ct_seg.data.loaders import create_dataloaders
from pinkcc_ct_seg.models.resnet18 import get_resnet18
from pinkcc_ct_seg.training.engine import train_one_epoch, validate
from pinkcc_ct_seg.evaluation.metrics import calculate_metrics

# 3. Configuration matérielle
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"💻 Appareil utilisé : {device}")

### 1. Chargement des données avec l'option `advanced`

In [ ]:
data_dir = project_root / "data" / "raw" / "brain_mri" / "Training"

# Création des splits (ajustez le random_state si besoin)
train_items, val_items = get_train_val_splits(data_dir, val_size=0.2, random_state=42)

train_dataset = BrainMRIDataset(train_items)
val_dataset = BrainMRIDataset(val_items)

# Application du raffinage des transformations
train_loader, val_loader = create_dataloaders(
    train_dataset, 
    val_dataset, 
    batch_size=32, 
    augmentation_level='advanced'
)

print(f"✅ Entraînement: {len(train_loader.dataset)} images | Validation: {len(val_loader.dataset)} images.")

### 2. Configuration du Modèle et de la Méthode B (Poids des classes)

In [ ]:
# --- MÉTHODE B : Gestion du déséquilibre ---
# ATTENTION : Remplacez par vos vrais effectifs de la classe 0 et 1
num_class_0 = 400  # Ex: Pas de tumeur
num_class_1 = 100  # Ex: Tumeur
total = num_class_0 + num_class_1

# Calcul des poids inversement proportionnels
weight_class_0 = total / (2.0 * num_class_0)
weight_class_1 = total / (2.0 * num_class_1)
class_weights = torch.tensor([weight_class_0, weight_class_1]).to(device)
print(f"⚖️ Poids de la Méthode B : Classe 0 = {weight_class_0:.2f}, Classe 1 = {weight_class_1:.2f}")

# --- INITIALISATION ---
model = get_resnet18(num_classes=2, pretrained=True).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(model.parameters(), lr=0.001)

### 3. Boucle d'entraînement

In [ ]:
EPOCHS = 10
train_losses, val_losses = [], []
best_val_loss = float('inf')

print("🚀 Début de l'entraînement...")

for epoch in range(EPOCHS):
    print(f"\n--- Époque {epoch+1}/{EPOCHS} ---")
    
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, y_true, y_pred = validate(model, val_loader, criterion, device)
    
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    
    print(f"📉 Train Loss: {train_loss:.4f} | 📈 Val Loss: {val_loss:.4f}")
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        save_path = project_root / "models" / "best_resnet18_methodB_advanced.pt"
        torch.save(model.state_dict(), save_path)
        print("💾 Nouveau meilleur modèle sauvegardé !")

### 4. Évaluation Finale des Performances

In [ ]:
# Recharger le meilleur modèle
model.load_state_dict(torch.load(project_root / "models" / "best_resnet18_methodB_advanced.pt"))

_, y_true_final, y_pred_final = validate(model, val_loader, criterion, device)
metrics_results = calculate_metrics(y_true_final, y_pred_final)

print("\n📊 --- RÉSULTATS GLOBAUX --- 📊")
print(f"Précision (Accuracy) : {metrics_results.get('accuracy', 0):.4f}")
print(f"F1-Score (Macro)     : {metrics_results.get('f1_score', 0):.4f}")
print(f"Rappel (Recall)      : {metrics_results.get('recall', 0):.4f}")

plt.figure(figsize=(10, 5))
plt.plot(range(1, EPOCHS+1), train_losses, label='Train Loss', color='blue')
plt.plot(range(1, EPOCHS+1), val_losses, label='Validation Loss', color='orange')
plt.xlabel('Époques')
plt.ylabel('Perte (Loss)')
plt.title('Courbe d\'apprentissage (Method B + Advanced Aug)')
plt.legend()
plt.grid(True)
plt.show()